In [ ]:
#Importaciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

print("Librerías cargadas.")

In [ ]:
Cargar dataset
df = pd.read_csv("../data/tickets_train.csv")
df.head()

In [ ]:
#Preparar variables (texto y features numéricos opcionales)
# Variables de entrada
text = df["ticket_text"].astype(str)

# Features adicionales (opcional pero recomendado)
extra_features = df[[
    "sentiment_label",
    "is_phishing",
    "has_pii",
    "project_age_months",
    "tickets_corrective_last_30d",
    "tickets_evolutionary_last_30d",
]]

# Target
y = df["ticket_type"].map({"Correctivo": 0, "Evolutivo": 1})

In [ ]:
#Vectoriar texto
vectorizer = TfidfVectorizer(max_features=3000)
X_text = vectorizer.fit_transform(text)

# Combinamos texto + numérico
X = np.hstack((X_text.toarray(), extra_features.values))
X.shape

In [ ]:
#Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
#Entrenar Modelo
model = LogisticRegression(max_iter=300)
model.fit(X_train, y_train)

print("Modelo entrenado.")

In [ ]:
#Evaluacion
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de Confusión")
plt.show()

In [ ]:
#Guardar Modelo
joblib.dump(model, "../models/model_ticket_type.pkl")
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")

print("Modelo y vectorizer guardados.")